# Plan view composites
This notebook plots spatial composites for each ensemble set. Select the ensemble set, and pick the times to composite across. Several cells are commented out, but are useful if you only want to look at composites of ensemble members that produced TCs...

Otherwise, we are compositing across an absolute time, which is the time at which TC genesis occurred in nature. This can be adjusted, but should be the same length as the init_times list, or the number of ensemble members.

### One important note...
The final composite is window is limited to that of the user's selection. Make sure you know the location of the wave whether Paulette or Rene, and base the window zoom on that preference. 

In [2]:
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os
import glob
from netCDF4 import Dataset

from metpy.units import units
import geocat.viz as gv
import geopy.distance


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker 

import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import wrf
from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)

import metpy.calc as mpcalc
import seaborn as sns

In [3]:
## The following are lists to select each ensemble set and respective initialization time
variation = ['fluxon', 'rst_on24', 'rst_on36', 'rst_on48', '', 'fluxoff', '', '', '']
vartitles  = ['control', 'noflux24hr', 'noflux36hr', 'noflux48hr', '', 'fluxoff', '', '', '']
titles = [
    'Control',
    '24 hours without fluxes',
    '36 hours without fluxes',
    '48 hours without fluxes',
    '',
    'Fluxes off',
    '',
    '',
    ''
]
init_times = ['0300','0303','0306',
             '0309','0312','0315',
             '0318','0321','0400']

In [4]:
#########################################
### MAKE SELECTIONS HERE
wave = 'second'             # 'second' == Paulette or 'main' == Rene
var_index = 1

set_name = variation[var_index]     # try: variation[1]
vartitle = vartitles[var_index]
title = titles[var_index]  # What will be displayed on main plot
ens = init_times[3]


## 
if wave == 'main':
    name = 'Rene'
else:
    name = 'Paulette'

In [5]:
save_name = set_name +"_"+ ens
print(save_name)
title_name = vartitle+"_"+ens
print(title_name)

rst_on24_0309
noflux24hr_0309


In [6]:
plotsdir = '/glade/u/home/athornton/qtrack/'

In [7]:
### This is the time a TC forms in this simulation
tc_times = ['2020-09-08 21:00:00'] ## 24
#tc_times = ['2020-09-09 09:00:00'] ## 36
#tc_times = ['2020-09-09 06:00:00'] ## 48
#tc_times = ['2020-09-07 12:00:00'] ## control
#tc_times = ['2020-09-08 09:00:00'] ## control 0303

#tc_times = ['2020-09-09 12:00:00'] ## end

In [11]:
os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/cent_atl_case/init"+ens+"z/"+set_name+"/")
datafiles = sorted(glob.glob("./wrfout*"))
numfiles=len(datafiles)
print(ens)
print(tc_times[0])
init_time = pd.to_datetime('2020-09-'+ens[0:2]+' '+ens[2:4]+':00:00')
end_time = pd.to_datetime('2020-09-09 12:00:00')
dates_list = pd.date_range(start=init_time, end=end_time, freq='3h')
date_index = np.squeeze(np.where(dates_list == tc_times[0]))

wrf_out_data = xr.open_dataset(datafiles[date_index])  
p=wrf_out_data['P'][0]
lats, lons = latlon_coords(p)
lats_np = to_np(lats)
lons_np = to_np(lons)

ncfile = Dataset(datafiles[date_index])
Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
timestr=(str(Time))
# Set up one time string for plot titles, another for file names
titletime=(timestr[0:10]+' '+timestr[11:16])
filetime=(timestr[0:10]+'_'+timestr[11:16])
print('WRF valid time: ',filetime)

wrf_out_data = xr.open_dataset(datafiles[date_index])  

qwv = wrf_out_data["QVAPOR"][0].metpy.quantify()
pb = wrf_out_data['PB'].metpy.quantify()
p = wrf_out_data['P'].metpy.quantify()

# Get all the variables we need
ua = wrf_out_data["U"]
va = wrf_out_data["V"]

# Unstagger the winds to match the pressure grid
ua = destagger(ua, stagger_dim=3)             # Unstagger U in the x-direction
va = destagger(va, stagger_dim=2)             # Unstagger V in the x-direction

# Grid spacing for vorticity calculation
dx = ncfile.DX * units("m")
dy = ncfile.DY * units("m")

u_winds = xr.DataArray(ua)[0]
v_winds = xr.DataArray(va)[0]

u_lay = u_winds.sel(dim_1=slice(12,19))*units('m/s') 
v_lay = v_winds.sel(dim_1=slice(12,19))*units('m/s')

### FOR VORTICITY
vo = mpcalc.vorticity(u_lay, v_lay, dx=dx, dy=dy).mean(dim='dim_1')
vo.name = 'vorticity'
#vortlayer = smooth2d(vo, 10, cenweight=8)

### FOR STREAMLINES
# u = u_winds.sel(dim_1=17)*units('m/s') 
# v = u_winds.sel(dim_1=17)*units('m/s') 
u = u_lay.mean(dim='dim_1')
v = v_lay.mean(dim='dim_1')
u = to_np(u)
v = to_np(v)

### FOR PRECIPITATION
datafiles_ens = datafiles[0:date_index]
data = xr.open_mfdataset(datafiles_ens, concat_dim='Time', combine='nested', engine='netcdf4')
rain = data.RAINC
for i in range(0,len(rain)):
    rain_hour = rain[i] - rain[i-1]
    # loop through til the end. final point is the correct point

0309
2020-09-08 21:00:00
WRF valid time:  2020-09-08_21:00


/glade/derecho/scratch/athornton/tmp/ipykernel_70165/955782304.py:50: UserWarning: Horizontal dimension numbers not found. Defaulting to (..., Y, X) order.
  vo = mpcalc.vorticity(u_lay, v_lay, dx=dx, dy=dy).mean(dim='dim_1')


In [12]:
# From Anantha...
# Simple linear interpolation between each latitude point
def get_lats():
    new_lats = []
    wave_track_lats = df.lat[-len(times_list):]
    print(len(wave_track_lats))
    for i in range(0,len(wave_track_lats)):
        if wave_track_lats[i] == wave_track_lats[-1]:
            #new_lats.append(wave_track_lats[i].values)
            new_lats.append(wave_track_lats[i].values)
        else:
            next_value = 0.5*(wave_track_lats[i] + wave_track_lats[i+1])
            new_lats.append(wave_track_lats[i].values)
            new_lats.append(next_value.values)
    return new_lats

# From Anantha...
# Simple linear interpolation between each longitude point
def get_lons():
    new_lons = []
    wave_track_lons = df.lon[-len(times_list):] 
    for i in range(0,len(wave_track_lons)):
        if wave_track_lons[i] == wave_track_lons[-1]:
            #new_lons.append(wave_track_lons[i].values)
            new_lons.append(wave_track_lons[i].values)
        else:
            next_value = 0.5*(wave_track_lons[i] + wave_track_lons[i+1])
            new_lons.append(wave_track_lons[i].values)
            new_lons.append(next_value.values)
    return new_lons

save_name = set_name +"_"+ ens

df = xr.open_dataset('/glade/u/home/athornton/qtrack/'+wave+'_wave/'+wave+'_wave_track_'+save_name+'.nc')
times_list = pd.date_range(start=init_time, end=end_time, freq='6h')

wave_track_lats = get_lats()
wave_track_lons = get_lons()

25


In [13]:
def get_marsupial():
    save_name = set_name +"_"+ ens

    df = xr.open_dataset('/glade/u/home/athornton/qtrack/'+wave+'_wave/'+wave+'_wave_track_'+save_name+'.nc')
    times_list = pd.date_range(start=init_time, end=end_time, freq='6h')
    
    wave_track_lats = get_lats()
    wave_track_lons = get_lons()
    ## Lat lon of TC genesis
    lat2 = wave_track_lats[date_index] # -1 if using an odd ens mem
    lon2 = wave_track_lons[date_index] # -1 if using an odd ens mem
    
    ## Previous lat lon
    lat1 = wave_track_lats[date_index-1] # -2 if using an odd ens mem
    lon1 = wave_track_lons[date_index-1] # -2 if using an odd ens mem
    
    lat = (lat1 + lat2)/2
    lon = (lon1 + lon2)/2
    
    ## For calculating u vector
    u_lat_lon = [lat2, lon2]
    u_lat_lon_prev = [lat, lon1]
    
    ## For calculating v vector
    v_lat_lon = [lat2, lon2]
    v_lat_lon_prev = [lat1, lon]
    
    u_dist = geopy.distance.geodesic(u_lat_lon_prev, u_lat_lon).m
    v_dist = geopy.distance.geodesic(v_lat_lon_prev, v_lat_lon).m
    
    u_mar = u_dist/10800
    v_mar = v_dist/10800

    return u_mar, v_mar

In [ ]:
# define plotting function
def create_fig():
    fig = plt.figure(figsize=(10,12))
    proj=ccrs.PlateCarree()
    
    ax = plt.axes(projection=proj)

    ######## MAKE SURE THIS IS THE WINDOW YOU WANT ###############
    # Paulette around TC genesis should be centered around 45 W
    if wave == 'second':
        ax.set_extent([-70, -40, 0, 30],ccrs.PlateCarree())
        #ax.set_extent([-70, -20, 0, 40],ccrs.PlateCarree())
        name = 'Paulette'
    # Rene around TC genesis should be centered around 25 W
    else: 
        ax.set_extent([-40, -10, 0, 30],ccrs.PlateCarree())
        name = 'Rene'

    ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=.5, edgecolor="black")
    ax.add_feature(cfeature.COASTLINE.with_scale('10m'), linewidth=0.8, edgecolor='black')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=1.0)
    
    #gv.set_titles_and_labels(ax, lefttitle=" 600 hPa Zonal Winds", lefttitlefontsize=18)
    plt.title('Precipitation, 850-600 vorticity, and 700 hPa streamlines', loc='left', weight='bold', fontsize=14)
    plt.title(title +'\n'+name+'\n'+filetime, loc='right', fontsize=14)
    gl1 = ax.gridlines(crs=proj, linewidth=1, color='darkgray', draw_labels=True)
    gl1.top_labels = False
    gl1.right_labels = False
    gl1.xlocator = mticker.FixedLocator(np.arange(-90,40,5))
    gl1.ylocator = mticker.FixedLocator(np.arange(-5,40,5))
    gl1.xformatter = LongitudeFormatter(degree_symbol='')
    gl1.yformatter = LatitudeFormatter(degree_symbol='')
    
    vortlevels = np.arange(-0.00020, 0.00020, 0.00002)
    
    vo_plot = ax.contourf(lons_np, lats_np, vo, 
                          levels=vortlevels, transform=proj,
                          extend='both', cmap='bwr')
    u_mar, v_mar = get_marsupial()
    
    ax.streamplot(lons_np, lats_np, u+u_mar,v-v_mar, density=0.4, broken_streamlines=False, color='black', linewidth=1.9)
    #ax.streamplot(lons_np, lats_np, u,v, density=0.4, broken_streamlines=False, color='darkgoldenrod', linewidth=1.8)

    preciplevels = np.arange(0.2, 1.2, 0.8)#0.2)
    precip_plot = ax.contourf(lons_np, lats_np, rain_hour, 
                          levels=preciplevels, transform=proj, cmap='binary', alpha=0.45, extend='max')
                          #extend='max', cmap='YlGnBu')
    plt.tight_layout()
    cb = plt.colorbar(vo_plot, ax=ax, orientation="horizontal", pad=.03, shrink=.8, aspect=30)
    cb.set_label('Vorticity ($1$/$s$)', fontsize=14)

    outTPlotName= 'planview_'+save_name+'_'+wave+'_TC.png'
    plt.savefig(plotsdir+outTPlotName, bbox_inches='tight')



In [ ]:
f = create_fig()

In [24]:
## Lat lon of TC genesis
lat2 = wave_track_lats[date_index]
lon2 = wave_track_lons[date_index]

## Previous lat lon
lat1 = wave_track_lats[date_index-1]
lon1 = wave_track_lons[date_index-1]

lat = (lat1 + lat2)/2
lon = (lon1 + lon2)/2

## For calculating u vector
u_lat_lon = [lat2, lon2]
u_lat_lon_prev = [lat, lon1]

## For calculating v vector
v_lat_lon = [lat2, lon2]
v_lat_lon_prev = [lat1, lon]

u_dist = geopy.distance.geodesic(u_lat_lon_prev, u_lat_lon).m
v_dist = geopy.distance.geodesic(v_lat_lon_prev, v_lat_lon).m

u_mar = u_dist/10800
v_mar = v_dist/10800

In [44]:
u_mar

0.3191876498344389

In [37]:
# lat_lon_prev = [lat1, lon1]
# lat_lon = [lat2, lon2]
# dist = geopy.distance.geodesic(lat_lon_prev, lat_lon).m
# speed = dist/10800 # 3 hours between track points
# mar_speed = np.sqrt(u_mar**2 + v_mar**2)

In [40]:
u_mar, v_mar = get_marsupial()

In [41]:
u_mar

0.3191876498344389

In [42]:
v_mar

0.16252313972761734